# Module 7 bis - Klaro - Notebook de pilotage

Ce notebook deroule le brief en binome : l'ancienne architecture, puis les
4 agents (2 par apprenant), tous bases sur un vrai modele Mistral. Aucune
generation de donnees : tout est statique dans `data/`.

Les TODO se trouvent dans les fichiers `src/agent/agent*.py`, pas dans ce
notebook : les cellules ci-dessous appellent simplement `repondre(...)` /
`executer_agent4(...)`, deja fournis. Tant qu'un agent n'est pas termine,
la cellule correspondante leve `NotImplementedError`.

> Avant de commencer : copiez `.env.example` en `.env` et renseignez
> `MISTRAL_API_KEY` (voir `README.md`). Chaque cellule ci-dessous declenche
> un ou plusieurs appels API reels.

## Bootstrap - remonter a la racine du projet

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import importlib
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config as C, data as D
from src.agent.ancien_bot import repondre_ancien_bot
from src.agent.tools import ContexteOutils
from src.agent import agent1_renseignement as A1
from src.agent import agent2_remboursement as A2
from src.agent import agent3_pretriage as A3
from src.agent import agent4_autonomie as A4
print("Racine:", ROOT)

Racine: /home/a453784/simplon-briefs/M7B1


In [2]:
import os
if not os.environ.get("MISTRAL_API_KEY"):
    print("MISTRAL_API_KEY absente : copiez .env.example en .env et renseignez votre cle "
         "(https://console.mistral.ai/) avant d'executer les cellules suivantes.")
else:
    print("MISTRAL_API_KEY detectee.")

MISTRAL_API_KEY detectee.


## Chargement des donnees

In [3]:
faq = D.charger_faq()
commandes = D.charger_commandes()
tickets = D.charger_tickets()
messages = D.charger_messages()
len(faq), len(commandes), len(tickets), len(messages)

(20, 45, 18, 50)

## Etape 1 - L'ancienne architecture (a evaluer, en binome)

Pas de code a completer ici : executez l'ancien bot sur quelques questions,
observez ses limites, et redigez a deux la note d'architecture et de risques
(livrable ecrit, voir le brief). C'est aussi ici que vous vous repartissez
les agents 1 a 4.

In [4]:
for q in ["Ou en est ma commande CMD-1013 ?",
         "Je veux etre rembourse pour la commande CMD-1007"]:
    print("Q:", q)
    print("R (ancien bot):", repondre_ancien_bot(q))
    print()

Q: Ou en est ma commande CMD-1013 ?
R (ancien bot): Je n'ai pas compris votre demande, un conseiller vous recontactera.

Q: Je veux etre rembourse pour la commande CMD-1007
R (ancien bot): Pas de souci, vous serez rembourse sous peu !



## Agent 1 - Renseignement general

FAQ + suivi de commande. 2 outils : `chercher_faq`, `consulter_commande`.
TODO dans `src/agent/agent1_renseignement.py` : `construire_agent`.

In [5]:
ctx = ContexteOutils(faq=faq, commandes=commandes, tickets=tickets)

for q in ["Ou en est ma commande CMD-1013 ?", "Quel est le delai de livraison standard ?"]:
    res = A1.repondre(ctx, q)
    print("Q:", q)
    print("R:", res["diagnostic"]["reponse"])
    print()

Q: Ou en est ma commande CMD-1013 ?
R: Votre commande **CMD-1013** pour l'**enceinte Bluetooth Sona** (79 €) est actuellement **en retard**. Elle était initialement prévue pour le **14 février 2026**.

Je vous tiens informé dès que nous avons une mise à jour concernant sa livraison. Si vous avez besoin d’assistance supplémentaire, n’hésitez pas à me le demander !

Q: Quel est le delai de livraison standard ?
R: Le délai de livraison standard pour Klaro est de **3 à 5 jours ouvrés** à compter de la validation de votre commande.



## Agent 2 - Remboursement

Verifie l'eligibilite, ne rembourse jamais seul (pas d'outil d'ecriture). 2
outils : `verifier_eligibilite_remboursement`, `rechercher_historique`.
TODO dans `src/agent/agent2_remboursement.py` : `construire_agent`.

In [6]:
importlib.reload(A2)
importlib.reload(C)

for q in ["Je veux etre rembourse pour la commande CMD-1004",
         "Je veux etre rembourse pour la commande CMD-1007"]:
    res = A2.repondre(ctx, q)
    print("Q:", q)
    print("R:", res["diagnostic"]["reponse"])
    print()

Q: Je veux etre rembourse pour la commande CMD-1004
R: La commande **CMD-1004** est **éligible à un remboursement partiel** selon les critères suivants :
- **Motif** : La commande a été livrée il y a **2 jours**, ce qui respecte le délai de **14 jours** pour une demande de remboursement.
- **Montant maximal éligible** : **24,90 €** (conformément à notre politique de remboursement standard pour les produits non utilisés et dans leur emballage d'origine).

### Prochaines étapes :
Un membre de notre équipe humaine devra valider cette demande et traiter le remboursement. Souhaitez-vous que je recherche un **précédent similaire** dans notre historique pour vous donner une idée du processus ?

*(Rappel : Le remboursement final dépend de la conformité de votre demande avec nos conditions générales.)*

Q: Je veux etre rembourse pour la commande CMD-1007
R: La commande **CMD-1007** ne semble pas éligible à un remboursement standard, car elle a été livrée **il y a 86 jours**, dépassant ainsi le 

## Agent 3 - Pre-triage des messages entrants

Mesurez le taux d'accord et la part de propositions validees sans
modification (biais d'automatisation). TODO dans
`src/agent/agent3_pretriage.py` : `classifier_message`.

`C.LIMITE_MESSAGES_PRETRIAGE` (dans `config.py`) borne le nombre de messages
traites pour economiser des appels API pendant le developpement.

In [7]:
pretri = A3.pretrier(messages)
valide = A3.simuler_validation(pretri)
A3.taux(valide)

{'taux_accord_global': 0.875,
 'taux_accord_ambigus': nan,
 'part_validee_sans_modification': 0.875,
 'alerte_biais_automatisation': True}

## Agent 4 - Autonomie sous contrainte

Propose puis, seulement si un humain approuve, execute un remboursement
automatise dans les cas les plus clairs (middleware human-in-the-loop). TODO
dans `src/agent/agent4_autonomie.py` : `construire_agent`.

Trois cas : un cas approuve, un cas rejete, un cas au-dessus du plafond
automatisable (refuse par l'outil quoi qu'il arrive).

In [8]:
print(A4.executer_agent4(ctx, "CMD-1004", {"type": "approve"})["diagnostic"]["reponse"])
print(A4.executer_agent4(ctx, "CMD-1035", {"type": "reject", "message": "non valide par le conseiller"})["diagnostic"]["reponse"])
print(A4.executer_agent4(ctx, "CMD-1009", {"type": "approve"})["diagnostic"]["reponse"])

La commande **CMD-1004** est **éligible** au remboursement pour les raisons suivantes :
- **Motif** : Livrée il y a 2 jours (dans le délai de 14 jours).
- **Montant total** : **24,90 EUR** (inférieur au seuil de 60,00 EUR).

### Proposition :
Je peux **initier le processus de remboursement** pour cette commande. Cependant, comme mentionné précédemment, ce déclenchement reste soumis à une **validation humaine explicite**. Dois-je procéder à l'initiation du remboursement en attente de cette validation ?
La commande **CMD-1035** est **éligible** au remboursement standard pour les raisons suivantes :
- **Livrée il y a 1 jour**, dans le délai de 14 jours.
- **Montant maximal éligible : 59,90 EUR** (inférieur ou égal à 60,0 EUR).

Je propose d'**initier le remboursement** pour ce montant. Dois-je procéder ? (⚠️ Rappel : Une validation humaine explicite reste nécessaire avant finalisation.)
La commande **CMD-1009** est **éligible** au remboursement pour les raisons suivantes :
- **Motif** : L

## Etape 3 - Evaluation croisee (en dehors de ce notebook)

Chacun teste et evalue les 2 agents developpes par son binome (pas les
siens) et redige un rapport court. Voir le brief, etape 3.

## Etape 4 - Integrer et decider (en binome)

Mettez en commun les 4 agents, verifiez qu'ils s'articulent sans se
contredire, et redigez une decision de deploiement commune a partir des deux
rapports d'evaluation croisee. Voir le brief, etape 4.